# Tetris OpenEnv — Per-Piece GRPO Training

Train an LLM to play Tetris: model sees the board before **every piece**, outputs actions until piece locks, learns via GRPO-style policy gradient.

**Environment**: Local Tetris engine (game_engine.py from OpenEnv)
**Model**: Qwen2.5-3B-Instruct + LoRA (r=16)
**Training**: Custom REINFORCE/GRPO loop — 8 games/iteration, 100 iterations
**Runtime**: L4 GPU (Colab)

In [1]:
# Cell 1: Install dependencies
!pip install peft accelerate -q
!pip install -U bitsandbytes -q
!pip install git+https://github.com/huggingface/transformers.git --quiet
!python -c "import bitsandbytes; print(bitsandbytes.__version__)"


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 41.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
0.49.2


In [2]:
# Cell 2: Load Qwen3.5-27B + LoRA (INT4 / QLoRA, NO thinking)
import gc
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model

gc.collect()
torch.cuda.empty_cache()

model_name = "Qwen/Qwen3.5-27B"

tokenizer = AutoTokenizer.from_pretrained(model_name)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)

# Gradient checkpointing — критично для 27B, экономит VRAM при backprop
model.gradient_checkpointing_enable()

lora_config = LoraConfig(
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# Проверка свободной памяти
free_mem = torch.cuda.mem_get_info()[0] / 1024**3
total_mem = torch.cuda.mem_get_info()[1] / 1024**3
print(f"\nGPU memory: {total_mem - free_mem:.1f}GB used / {total_mem:.1f}GB total ({free_mem:.1f}GB free)")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/12.8M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

...

Fetching 11 files:   0%|          | 0/11 [00:00<?, ?it/s]

The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/851 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/244 [00:00<?, ?B/s]

trainable params: 79,691,776 || all params: 26,975,690,240 || trainable%: 0.2954

GPU memory: 48.3GB used / 95.0GB total (46.7GB free)


In [11]:
# Cell 3: Game engine + constants + logit-masked generation (Qwen3.5-27B)
import random
import torch.nn.functional as F

# Download game engine from repo
!wget -q -O game_engine.py "https://raw.githubusercontent.com/OutOfMystic/tetris-openenv/main/src/tetris_env/server/game_engine.py?$(date +%s)"
from game_engine import TetrisEnv, __version__
print(f"Game engine v{__version__}")

# === Constants ===
MAX_ACTIONS_PER_PIECE = 20
MAX_PIECES_PER_GAME = 200
GAMES_PER_ITER = 8
NUM_ITERATIONS = 100
TEMPERATURE = 0.8

# Training reward modifiers
LR_PENALTY = 0.0
PIECE_PLACED_BONUS = 2.0
NO_PLACE_PENALTY = -10.0
LINE_CLEAR_BONUS = 100.0

# Action mapping — semantic words
ACTION_CHARS = ['Left', 'Right', 'Rotate', 'Counter', 'Drop', 'Down']
ACTION_TO_ENGINE = {
    'Left': 'left', 'Right': 'right', 'Rotate': 'rotate_cw',
    'Counter': 'rotate_ccw', 'Drop': 'drop', 'Down': 'down'
}

# Pre-compute token IDs
ACTION_TOKEN_IDS = []
for word in ACTION_CHARS:
    ids = tokenizer.encode(word, add_special_tokens=False)
    assert len(ids) == 1, f"'{word}' is {len(ids)} tokens, expected 1!"
    ACTION_TOKEN_IDS.append(ids[0])
    print(f"  '{word}' -> token_id {ids[0]}")
ACTION_TOKEN_IDS = torch.tensor(ACTION_TOKEN_IDS, device=model.device)

TOKEN_TO_CHAR = {ACTION_TOKEN_IDS[i].item(): ACTION_CHARS[i] for i in range(6)}

SYSTEM_PROMPT = """You play Tetris on a 10-wide, 20-tall board.
The falling piece '@' starts at the top-center. Locked blocks are '#'.

Available moves (one word each):
  Left    — shift piece 1 column left
  Right   — shift piece 1 column right
  Rotate  — rotate 90° clockwise
  Counter — rotate 90° counter-clockwise
  Down    — soft drop 1 row
  Drop    — hard drop (piece falls to bottom and locks)

After every move except Drop, gravity pulls the piece down 1 row.

Goal: Fill complete rows. Keep the board flat. Avoid holes.

Respond ONLY with move words separated by spaces. End with Drop."""


def build_prompt(result):
    return f"""Board:
{result['board']}

Piece: {result['current_piece']}  Shape:
{result['current_piece_shape']}
Position: column {result['piece_x']}, row {result['piece_y']}
Next: {result['next_piece']}
Score: {result['score']}  Lines: {result['total_lines']}  Holes: {result['holes']}  Height: {result['max_height']}

Your moves:"""


# Compute bias correction on Qwen3.5-27B
with torch.no_grad():
    _env = TetrisEnv(seed=0)
    _env.reset(seed=0)
    _msgs = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": build_prompt(_env._make_result(0))},
    ]
    _text = tokenizer.apply_chat_template(
        _msgs, tokenize=False, add_generation_prompt=True,
        enable_thinking=False
    )
    _ids = tokenizer(_text, return_tensors="pt").input_ids.to(model.device)
    _logits = model(_ids).logits[0, -1, ACTION_TOKEN_IDS]
    ACTION_BIAS = _logits.mean() - _logits
    print("\nRaw logits for action tokens:")
    for i, word in enumerate(ACTION_CHARS):
        print(f"  {word:10s}: logit={_logits[i].item():+.2f}  bias={ACTION_BIAS[i].item():+.2f}")

print(f"\nReady. Model: Qwen3.5-27B, logit masking ON, thinking OFF")


Game engine v0.6.0
  'Left' -> token_id 5247
  'Right' -> token_id 5791
  'Rotate' -> token_id 33395
  'Counter' -> token_id 13695
  'Drop' -> token_id 19266
  'Down' -> token_id 4308

Raw logits for action tokens:
  Left      : logit=+22.50  bias=-0.88
  Right     : logit=+21.38  bias=+0.25
  Rotate    : logit=+21.62  bias=+0.00
  Counter   : logit=+21.25  bias=+0.38
  Drop      : logit=+22.12  bias=-0.50
  Down      : logit=+21.00  bias=+0.62

Ready. Model: Qwen3.5-27B, logit masking ON, thinking OFF


In [12]:
# Cell 4: Core training functions — sequential play (1 game at a time)
import time as _time

def play_one_game(model, tokenizer, seed, temperature=TEMPERATURE):
    """Play a single Tetris game. Returns game result for GRPO training."""
    env = TetrisEnv(seed=seed)
    env.reset(seed=seed)

    rng = random.Random(seed)
    moves = rng.randint(0, 4)
    direction = rng.choice(["left", "right"])
    for _ in range(moves):
        if env.done:
            break
        env.step(direction)

    total_reward = 0.0
    total_steps = 0
    pieces_count = 0
    pieces_data = []

    while not env.done and pieces_count < MAX_PIECES_PER_GAME:
        pieces_count += 1
        current_piece_name = env.current_piece_name
        lines_before = env.total_lines

        result = env._make_result(0)
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": build_prompt(result)},
        ]
        prompt_ids = tokenizer.apply_chat_template(
            messages, return_tensors="pt", add_generation_prompt=True
        )
        if hasattr(prompt_ids, 'input_ids'):
            prompt_ids = prompt_ids['input_ids']
        prompt_ids = prompt_ids.to(model.device)

        with torch.no_grad():
            out = model(prompt_ids, use_cache=True)
            past_kv = out.past_key_values

        action_ids = []
        piece_placed = False

        with torch.no_grad():
            for step in range(MAX_ACTIONS_PER_PIECE):
                logits = out.logits[:, -1, :]

                # Mask to action tokens + bias correction
                masked = torch.full_like(logits, float('-inf'))
                masked[:, ACTION_TOKEN_IDS] = logits[:, ACTION_TOKEN_IDS] + ACTION_BIAS
                probs = F.softmax(masked / temperature, dim=-1)

                token_id = torch.multinomial(probs, 1)
                tid = token_id.item()
                action_ids.append(tid)

                action_char = TOKEN_TO_CHAR[tid]
                action_name = ACTION_TO_ENGINE[action_char]
                step_result = env.step(action_name)

                total_reward += step_result['reward']
                total_steps += 1

                if env.current_piece_name != current_piece_name:
                    piece_placed = True
                    total_reward += PIECE_PLACED_BONUS
                    lc = env.total_lines - lines_before
                    if lc > 0:
                        total_reward += lc * LINE_CLEAR_BONUS
                    break

                if env.done:
                    break

                out = model(
                    token_id,
                    past_key_values=past_kv,
                    use_cache=True,
                )
                past_kv = out.past_key_values

        if not piece_placed and not env.done:
            env.step('drop')
            total_reward += NO_PLACE_PENALTY
            total_steps += 1
            lc = env.total_lines - lines_before
            if lc > 0:
                total_reward += lc * LINE_CLEAR_BONUS

        if action_ids:
            pieces_data.append({
                'prompt_ids': prompt_ids.cpu(),
                'action_ids': torch.tensor(action_ids, dtype=torch.long),
                'piece_name': current_piece_name,
            })

    return {
        'reward': total_reward,
        'pieces': pieces_data,
        'total_steps': total_steps,
        'total_lines': env.total_lines,
        'pieces_placed': pieces_count,
        'final_board': env.board_to_text(),
    }


def train_one_iteration(model, optimizer, seed, temperature=TEMPERATURE):
    """One GRPO iteration: play 8 games sequentially, compute advantages, update."""
    # Phase 1: Sequential rollout
    t_rollout = _time.time()
    games = []
    for g in range(GAMES_PER_ITER):
        game = play_one_game(model, tokenizer, seed, temperature)
        games.append(game)
    t_rollout = _time.time() - t_rollout

    rewards = torch.tensor([g['reward'] for g in games], dtype=torch.float32)
    mean_r = rewards.mean().item()
    std_r = rewards.std().item()

    if std_r < 1e-8:
        return {'mean_reward': mean_r, 'std_reward': 0.0, 'loss': 0.0,
                'avg_steps': sum(g['total_steps'] for g in games) / GAMES_PER_ITER,
                'avg_lines': sum(g['total_lines'] for g in games) / GAMES_PER_ITER,
                'avg_pieces': sum(g['pieces_placed'] for g in games) / GAMES_PER_ITER,
                't_rollout': t_rollout, 't_update': 0.0}

    advantages = ((rewards - rewards.mean()) / (rewards.std() + 1e-8)).tolist()

    # Phase 2: Gradient update (NO bias correction here — raw log_probs)
    t_update = _time.time()
    optimizer.zero_grad()
    total_pieces = max(1, sum(len(g['pieces']) for g in games))
    total_loss = 0.0

    for game_idx, game in enumerate(games):
        adv = advantages[game_idx]
        for piece in game['pieces']:
            prompt = piece['prompt_ids'].to(model.device)
            actions = piece['action_ids'].to(model.device)
            if len(actions) == 0:
                continue

            full_input = torch.cat([prompt.squeeze(0), actions]).unsqueeze(0)
            logits = model(full_input).logits

            P = prompt.shape[-1]
            action_logits = logits[0, P-1 : P-1+len(actions), :]

            masked = torch.full_like(action_logits, float('-inf'))
            masked[:, ACTION_TOKEN_IDS] = action_logits[:, ACTION_TOKEN_IDS]
            log_probs = F.log_softmax(masked / temperature, dim=-1)

            selected = log_probs.gather(1, actions.unsqueeze(1)).squeeze(1)
            piece_loss = -(selected.sum() * adv) / total_pieces

            piece_loss.backward()
            total_loss += piece_loss.item()

    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()
    t_update = _time.time() - t_update

    return {
        'mean_reward': mean_r,
        'std_reward': std_r,
        'loss': total_loss,
        'avg_steps': sum(g['total_steps'] for g in games) / GAMES_PER_ITER,
        'avg_lines': sum(g['total_lines'] for g in games) / GAMES_PER_ITER,
        'avg_pieces': sum(g['pieces_placed'] for g in games) / GAMES_PER_ITER,
        't_rollout': t_rollout,
        't_update': t_update,
    }

# Smoke test
print("Smoke test: playing 1 game...")
test_game = play_one_game(model, tokenizer, seed=0)
print(f"Reward: {test_game['reward']:.1f}, Steps: {test_game['total_steps']}, "
      f"Pieces: {test_game['pieces_placed']}, Lines: {test_game['total_lines']}")
print("Training functions ready.")


Smoke test: playing 1 game...
Reward: -465.9, Steps: 79, Pieces: 11, Lines: 0
Training functions ready.


In [15]:
# Cell 5: Demo — UNTRAINED model with full generation log
print("=== UNTRAINED MODEL ===\n")

game = play_one_game(model, tokenizer, seed=42)

for i, piece in enumerate(game['pieces']):
    actions = [TOKEN_TO_CHAR[tid.item()] for tid in piece['action_ids']]
    print(f"  Piece {i+1} ({piece['piece_name']}): {' '.join(actions)}")

print(f"\nPieces: {game['pieces_placed']}/{MAX_PIECES_PER_GAME}")
print(f"Total actions: {game['total_steps']}")
print(f"Lines cleared: {game['total_lines']}")
print(f"Game reward: {game['reward']:+.1f}")
print(f"\nFinal board:")
print(game['final_board'])
# Cell 5: Verify board updates between pieces
print("=== BOARD VISIBILITY CHECK ===\n")

env = TetrisEnv(seed=42)
env.reset(seed=42)



=== UNTRAINED MODEL ===

  Piece 1 (L): Right Down Rotate Right Down Drop
  Piece 2 (I): Right Right Right Right Down Down Down Down Down Down Down Down Down Down Down Down Down Down Down Down
  Piece 3 (L): Right Right Right Right Right Right Right Right Right Right Right Right Right Right Right
  Piece 4 (T): Right Right Down Down Down Down Down Down
  Piece 5 (O): Right Right Down Right Right Down Right Right Down Right Right Down Right Right Down Right Right Down Right Right
  Piece 6 (L): Right Right Down Down Down
  Piece 7 (I): Right Right Right Right Right Rotate Down
  Piece 8 (L): Right Right Rotate Down Down Down Down Right Down Down Down Down
  Piece 9 (Z): Right Rotate Counter
  Piece 10 (I): Right Rotate Counter Down Left Down Down Down Down
  Piece 11 (Z): Right

Pieces: 11/200
Total actions: 107
Lines cleared: 0
Game reward: -435.6

Final board:
+----------+
|.....##...|
|......##..|
|.....##...|
|......##..|
|......###.|
|......#...|
|......####|
|......#...|
|......#.

{'board': '+----------+\n|....@.....|\n|....@.....|\n|....@@....|\n|..........|\n|..........|\n|..........|\n|..........|\n|..........|\n|..........|\n|..........|\n|..........|\n|..........|\n|..........|\n|..........|\n|..........|\n|..........|\n|..........|\n|..........|\n|..........|\n|..........|\n+----------+',
 'current_piece': 'L',
 'current_piece_shape': '#.\n#.\n##',
 'next_piece': 'I',
 'next_piece_shape': '####',
 'piece_x': 4,
 'piece_y': 0,
 'score': 0,
 'total_lines': 0,
 'steps': 0,
 'max_height': 0,
 'holes': 0,
 'reward': 0,
 'done': False}

In [ ]:
# Cell 6: Two-phase curriculum training
import time

# Override: no LR penalty, higher exploration
LR_PENALTY = 0.0
TEMPERATURE = 1.0

optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=1e-4,
    weight_decay=0.01,
)

history = []

# =============================================
# PHASE 1: Learn to place pieces (5 pieces max, 50 iterations)
# =============================================
MAX_PIECES_PER_GAME = 5
NUM_ITERATIONS_P1 = 50

print("=" * 60)
print(f"PHASE 1: Learning basics — max {MAX_PIECES_PER_GAME} pieces, {NUM_ITERATIONS_P1} iterations")
print(f"LR_PENALTY={LR_PENALTY}, TEMPERATURE={TEMPERATURE}")
print("=" * 60)

for iteration in range(NUM_ITERATIONS_P1):
    t0 = time.time()
    stats = train_one_iteration(model, optimizer, seed=iteration, temperature=TEMPERATURE)
    t1 = time.time()
    history.append(stats)

    print(f"[P1 {iteration:3d}] "
          f"reward={stats['mean_reward']:+8.1f} "
          f"std={stats['std_reward']:6.1f} "
          f"loss={stats['loss']:7.3f} "
          f"steps={stats['avg_steps']:5.1f} "
          f"lines={stats['avg_lines']:4.1f} "
          f"pieces={stats['avg_pieces']:4.1f} "
          f"t={t1-t0:.0f}s")

print(f"\nPhase 1 complete! Final avg reward: {history[-1]['mean_reward']:+.1f}")

# =============================================
# PHASE 2: Full game (200 pieces max, 200 iterations)
# =============================================
MAX_PIECES_PER_GAME = 200
NUM_ITERATIONS_P2 = 100
TEMPERATURE = 0.8

print("\n" + "=" * 60)
print(f"PHASE 2: Full game — max {MAX_PIECES_PER_GAME} pieces, {NUM_ITERATIONS_P2} iterations")
print("=" * 60)

for iteration in range(NUM_ITERATIONS_P2):
    t0 = time.time()
    stats = train_one_iteration(model, optimizer, seed=NUM_ITERATIONS_P1 + iteration, temperature=TEMPERATURE)
    t1 = time.time()
    history.append(stats)

    print(f"[P2 {iteration:3d}] "
          f"reward={stats['mean_reward']:+8.1f} "
          f"std={stats['std_reward']:6.1f} "
          f"loss={stats['loss']:7.3f} "
          f"steps={stats['avg_steps']:5.1f} "
          f"lines={stats['avg_lines']:4.1f} "
          f"pieces={stats['avg_pieces']:4.1f} "
          f"t={t1-t0:.0f}s")

print(f"\nPhase 2 complete! Final avg reward: {history[-1]['mean_reward']:+.1f}")
print("Training complete!")


PHASE 1: Learning basics — max 5 pieces, 50 iterations
LR_PENALTY=0.0, TEMPERATURE=1.0
[P1   0] reward=  -169.7 std=  62.1 loss=  0.438 steps= 47.2 lines= 0.0 pieces= 5.0 t=68s
[P1   1] reward=  -306.4 std=  89.8 loss=  0.047 steps= 59.4 lines= 0.0 pieces= 5.0 t=73s
[P1   2] reward=  -323.0 std= 106.4 loss= -0.072 steps= 61.4 lines= 0.0 pieces= 5.0 t=75s


In [ ]:
# Cell 7: Plot reward curve
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

iters = range(len(history))

axes[0].plot(iters, [h['mean_reward'] for h in history])
axes[0].set_title('Mean Reward per Iteration')
axes[0].set_xlabel('Iteration')
axes[0].set_ylabel('Reward')

axes[1].plot(iters, [h['avg_lines'] for h in history])
axes[1].set_title('Avg Lines Cleared')
axes[1].set_xlabel('Iteration')

axes[2].plot(iters, [h['loss'] for h in history])
axes[2].set_title('Policy Gradient Loss')
axes[2].set_xlabel('Iteration')

plt.tight_layout()
plt.savefig('reward_curve.png', dpi=150)
plt.show()
print("Reward curve saved to reward_curve.png")

In [ ]:
# Cell 8: Demo — TRAINED model plays 3 games
print("=== TRAINED MODEL ===\n")

trained_rewards = []
for seed in [42, 123, 7]:
    game = play_one_game(model, tokenizer, seed=seed)
    print(f"Seed {seed}: reward={game['reward']:+.1f}, "
          f"steps={game['total_steps']}, lines={game['total_lines']}, "
          f"pieces={game['pieces_placed']}")

    # Show first 3 pieces' actions
    for j, piece in enumerate(game['pieces'][:3]):
        actions = [TOKEN_TO_CHAR[tid.item()] for tid in piece['action_ids']]
        print(f"    Piece {j+1} ({piece['piece_name']}): {' '.join(actions)}")

    trained_rewards.append(game['reward'])
    print()

avg_trained = sum(trained_rewards) / len(trained_rewards)
print(f"{'='*50}")
print(f"TRAINED avg reward (3 games): {avg_trained:+.1f}")
print(f"Avg lines: {sum(g['total_lines'] for g in [game]):.1f}")
print('='*50)


In [ ]:
# Cell 9: Push trained model to HF Hub
model.push_to_hub("VortexedSquirrel/tetris-agent-grpo")
tokenizer.push_to_hub("VortexedSquirrel/tetris-agent-grpo")
print("Model pushed to https://huggingface.co/VortexedSquirrel/tetris-agent-grpo")